# Notebook 3 (bonus): Key instance detection (KID)

Some multi-instance models don't just predict a property - they also learn *how much each conformer contributed*
to that prediction. In a multi-conformer model, this means the model can point at the specific conformer it thinks
is responsible for the molecule's activity - often called the **active conformer**. Finding that conformer is
what we call **key instance detection (KID)**.

This notebook shows how to get those per-conformer weights out of a trained model, how to check whether they
actually point at the right conformer, and how to look at them visually.

**Before running this notebook**, install two extra packages that aren't part of core QSARmil (they're only needed
for this bonus notebook):

```bash
pip install huggingface_hub py3Dmol
```

In [20]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "" # uncomment if you do not have GPU

In [1]:
import pickle

import numpy as np
from rdkit import Chem

### 1. Load a dataset with known "correct answers"

To check whether a model's guesses about important conformers are actually right, we need a dataset where we
already know the true answer. Here we use a small, purpose-built benchmark: each molecule's bag contains up to 20
conformers, and a handful of them were deliberately designed to match specific pharmacophore patterns (simple 3D
"triggers" for activity). Those matching conformers are the true **key instances**; the rest are not.

The more pharmacophore patterns a conformer matches, the more "active" it's considered. Each molecule's target
value is simply the highest number of patterns matched by any of its conformers (from 1 to 7).

We download this dataset, plus two small helper scripts this notebook uses, from a Hugging Face dataset repository
(they're not part of the `qsarmil` package itself, since they're specific to this one demo).

In [3]:
import importlib.util
from huggingface_hub import hf_hub_download

In [6]:
REPO_ID = "KagakuLab/QSARmil"

def load_module_from_hf(repo_id, filename, module_name, repo_type="dataset"):
    """Download a .py file from an HF repo and import it as a module, without touching sys.path."""
    path = hf_hub_download(repo_id, filename=filename, repo_type=repo_type)
    spec = importlib.util.spec_from_file_location(module_name, path)
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module

def split_into_conformers(mol):
    """Split an already-embedded multi-conformer Mol into a bag of single-conformer Mol objects."""
    bag = []
    for conf in mol.GetConformers():
        conf_mol = Chem.Mol(mol)
        conf_mol.RemoveAllConformers()
        conf_mol.AddConformer(conf, assignId=True)
        bag.append(conf_mol)
    return bag

In [7]:
pkl_path = hf_hub_download(REPO_ID, filename="notebooks/train_conf.pkl", repo_type="dataset")
with open(pkl_path, "rb") as f:
    data_train = pickle.load(f)

pkl_path = hf_hub_download(REPO_ID, filename="notebooks/test_conf.pkl", repo_type="dataset")
with open(pkl_path, "rb") as f:
    data_test = pickle.load(f)
    
len(data_train), len(data_test)

(1317, 356)

In [8]:
mols_train = [i[1] for i in data_train]
mols_test = [i[1] for i in data_test]

In [9]:
confs_train = [split_into_conformers(i) for i in mols_train]
confs_test = [split_into_conformers(i) for i in mols_test]

In [11]:
# target property (highest number of matched pharmacophores in the bag, from 1 to 7)
y_train = [i[3] for i in data_train]
y_test = [i[3] for i in data_test]

In [12]:
# ground-truth key conformer indices for each molecule - this is what we'll check the model's guesses against
idx_train = [i[2] for i in data_train]
idx_test = [i[2] for i in data_test]

### 2. Computing descriptors

Same idea as in Notebook 2: each conformer needs to become a list of numbers before a model can learn from it. Any
3D descriptor works here in principle, but for KID specifically it helps to pick one that changes noticeably from
one conformer to another (otherwise there's nothing for the model to tell them apart by) and that's actually
related to the property you're predicting.

In [13]:
from qsarmil.descriptor.wrapper import DescriptorWrapper
from molfeat.calc import Pharmacophore3D
from milearn.preprocessing import BagMinMaxScaler

In [14]:
desc_calc = DescriptorWrapper(Pharmacophore3D(factory="pmapper"), verbose=True)

In [15]:
x_train = desc_calc.run(confs_train)

Calculating descriptors: 1317/1317

In [16]:
x_test = desc_calc.run(confs_test)

Calculating descriptors: 356/356

In [17]:
scaler = BagMinMaxScaler()
scaler.fit(x_train)
x_train_scaled = scaler.transform(x_train)
x_test_scaled = scaler.transform(x_test)

### 3. Training a model that can explain its own predictions

Not every multi-instance method can do KID - only the ones that use an internal weighting mechanism to combine
conformers. QSARmil has a few of these: the attention-based
networks (`AdditiveAttentionNetwork`, `SelfAttentionNetwork`, `HopfieldAttentionNetwork`) and
`DynamicPoolingNetwork`. All of them offer, in addition to the usual `model.predict(x)`:

- **`model.get_instance_weights(x)`** - returns, for each molecule, one weight per conformer in its bag. A higher
  weight means the model considered that conformer more important for its prediction.

We'll use `DynamicPoolingNetwork` here.

In [18]:
from milearn.network.module.hopt import DEFAULT_PARAM_GRID
from milearn.network.regressor import (
                                       BagNetworkRegressor,
                                       InstanceNetworkRegressor,
                                       AdditiveAttentionNetworkRegressor,
                                       SelfAttentionNetworkRegressor,
                                       HopfieldAttentionNetworkRegressor,
                                       DynamicPoolingNetworkRegressor,
                                      )

In [24]:
model = DynamicPoolingNetworkRegressor(accelerator="cpu")  # small max_epochs, just to keep this notebook fast
model.hopt(x_train_scaled, y_train, param_grid=DEFAULT_PARAM_GRID, verbose=True)
model.fit(x_train_scaled, y_train)

Optimizing hyperparameter: activation (5 options)
[1/23 |  4.3% |  0.8 min] Value: relu, Epochs: 39, Loss: 0.0122
[2/23 |  8.7% |  0.9 min] Value: leakyrelu, Epochs: 56, Loss: 0.0102
[3/23 | 13.0% |  1.0 min] Value: gelu, Epochs: 66, Loss: 0.0089
[4/23 | 17.4% |  0.7 min] Value: elu, Epochs: 31, Loss: 0.0306
[5/23 | 21.7% |  0.9 min] Value: silu, Epochs: 45, Loss: 0.0102
Best activation = gelu, val_loss = 0.0089
Optimizing hyperparameter: learning_rate (2 options)
[6/23 | 26.1% |  0.4 min] Value: 0.0001, Epochs: 84, Loss: 0.0146
[7/23 | 30.4% |  0.3 min] Value: 0.001, Epochs: 47, Loss: 0.0097
Best learning_rate = 0.001, val_loss = 0.0097
Optimizing hyperparameter: batch_size (3 options)
[8/23 | 34.8% |  0.4 min] Value: 32, Epochs: 25, Loss: 0.0091
[9/23 | 39.1% |  0.4 min] Value: 512, Epochs: 63, Loss: 0.0138
[10/23 | 43.5% |  0.4 min] Value: 1024, Epochs: 60, Loss: 0.0110
Best batch_size = 32, val_loss = 0.0091
Optimizing hyperparameter: weight_decay (5 options)
[11/23 | 47.8% |  0.8 

DynamicPoolingNetworkRegressor(
  (instance_transformer): Sequential(
    (0): Linear(in_features=2048, out_features=2048, bias=True)
    (1): GELU(approximate='none')
    (2): Linear(in_features=2048, out_features=1024, bias=True)
    (3): GELU(approximate='none')
    (4): Linear(in_features=1024, out_features=512, bias=True)
    (5): GELU(approximate='none')
    (6): Linear(in_features=512, out_features=256, bias=True)
    (7): GELU(approximate='none')
    (8): Linear(in_features=256, out_features=128, bias=True)
    (9): GELU(approximate='none')
    (10): Linear(in_features=128, out_features=64, bias=True)
    (11): GELU(approximate='none')
  )
  (bag_estimator): Norm()
  (dynamic_pooling): DynamicPooling()
)

### 4. Checking whether the model's weights point at the right conformer

`kid_accuracy` (another small helper from the same Hugging Face repository) compares a model's predicted weights
against the true key conformers. It expects, for each molecule:

- `y_true`: a list of 0s and 1s, one per conformer, where `1` marks a true key conformer.
- `y_pred`: a list of the same length, with the model's predicted weight for each conformer.

It reports two numbers:
- **KID accuracy**: how often the single highest-weighted conformer (or the top `top_n`, if you ask for more than
  one) is actually a true key conformer.
- **Baseline accuracy**: what you'd get by picking conformers *at random* instead - useful for telling whether the
  model is actually better than guessing.

In [25]:
from sklearn.metrics import r2_score

In [26]:
def idx_to_binary(bags, key_indices):
    """Turn a list of key-conformer indices into a 0/1 label per conformer, one array per bag."""
    labels = []
    for bag, keys in zip(bags, key_indices):
        label = np.zeros(len(bag), dtype=int)
        label[keys] = 1
        labels.append(label)
    return labels

In [27]:
metrics = load_module_from_hf(REPO_ID, filename="notebooks/metrics.py", module_name="metrics")
kid_accuracy = metrics.kid_accuracy
keys_test = idx_to_binary(confs_test, idx_test)

In [28]:
y_pred = model.predict(x_test_scaled)
w_pred = model.get_instance_weights(x_test_scaled)
w_pred = [w.flatten() for w in w_pred]

top_n = 1

print(f"All molecules: {len(y_test)}")
print(f"Prediction accuracy (R2): {r2_score(y_test, y_pred):.2f}")

acc, exp = kid_accuracy(keys_test, w_pred, top_n=top_n)
print(f"KID accuracy: {acc:.2f}")
print(f"KID baseline accuracy (random guessing): {exp:.2f}")

All molecules: 356
Prediction accuracy (R2): 0.90
KID accuracy: 0.21
KID baseline accuracy (random guessing): 0.11


### 5. Looking at the conformers directly

Numbers only tell you so much - it can help a lot to actually *look* at the conformers a model picked. The helper
below (from the same Hugging Face repository, needs `py3Dmol` as noted at the top of this notebook) draws a grid of
3D views: the true key conformer(s) highlighted in red, and the model's top-weighted guess(es) highlighted in
blue.

In [29]:
visualization = load_module_from_hf(REPO_ID, filename="notebooks/visualization.py", module_name="visualization")
visualize_conformers_grid = visualization.visualize_conformers_grid

In [40]:
N = 2  # pick a molecule index from the test set to inspect

print("predicted:", y_pred[N], "| true:", y_test[N])
visualize_conformers_grid(mols_test[N], w_pred[N], idx_test[N], top_n=3, sort_by_weight=True)

predicted: 3.955306 | true: 4
